In [17]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader

In [18]:
class CNN(nn.Module):
    def __init__(self, num_filters1, num_filters2, kernel_size, dropout_rate, fc_units):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, num_filters1, kernel_size, padding=kernel_size//2)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(num_filters1, num_filters2, kernel_size, padding=kernel_size//2)
        self.dropout = nn.Dropout(dropout_rate)

        fc_input_size = num_filters2 * 8 * 8
        self.fc1 = nn.Linear(fc_input_size, fc_units)
        self.fc2 = nn.Linear(fc_units, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [19]:
def load_data():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    testloader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

    return trainloader, testloader

In [20]:
def train_and_evaluate(model, trainloader, testloader, optimizer, epochs=5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for i, data in enumerate(trainloader, 0):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 100 == 99:
                print(f'Epoch {epoch + 1}, Batch {i + 1}: Loss = {running_loss / 100:.3f}')
                running_loss = 0.0

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Test accuracy: {accuracy:.2f}%')
    return accuracy

In [21]:
def objective(trial):
    # Гиперпараметры
    num_filters1 = trial.suggest_int('num_filters1', 24, 128)                   # 1) количество фильтров в первом слое
    num_filters2 = trial.suggest_int('num_filters2', 48, 256)                   # 2) количество фильтров во втором слое
    kernel_size = trial.suggest_categorical('kernel_size', [3, 5, 7])           # 3) размер ядра свертки
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)                # 4) коэффициент дропаута
    fc_units = trial.suggest_int('fc_units', 128, 512)                          # 5) количество нейронов в полносвязном слое
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)  # 6) скорость обучения
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)    # 7) L2-регуляризация

    trainloader, testloader = load_data()
    model = CNN(num_filters1, num_filters2, kernel_size, dropout_rate, fc_units)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    accuracy = train_and_evaluate(model, trainloader, testloader, optimizer)

    return accuracy

In [22]:
study = optuna.create_study(direction='maximize')  # Максимизируем точность
study.optimize(objective, n_trials=6)              # 6 попыток поиска

print("Оптимальные гиперпараметры:")
for key, value in study.best_params.items():
    print(f"{key}: {value}")
print(f"Наилучшая точность: {study.best_value:.2f}%")

[I 2025-04-25 18:39:30,753] A new study created in memory with name: no-name-1b8f9f11-2910-4018-b8a5-f53d227021f7


Epoch 1, Batch 100: Loss = 2.224
Epoch 1, Batch 200: Loss = 1.953
Epoch 1, Batch 300: Loss = 1.895
Epoch 1, Batch 400: Loss = 1.856
Epoch 1, Batch 500: Loss = 1.858
Epoch 1, Batch 600: Loss = 1.830
Epoch 1, Batch 700: Loss = 1.807
Epoch 2, Batch 100: Loss = 1.796
Epoch 2, Batch 200: Loss = 1.792
Epoch 2, Batch 300: Loss = 1.786
Epoch 2, Batch 400: Loss = 1.817
Epoch 2, Batch 500: Loss = 1.796
Epoch 2, Batch 600: Loss = 1.781
Epoch 2, Batch 700: Loss = 1.769
Epoch 3, Batch 100: Loss = 1.758
Epoch 3, Batch 200: Loss = 1.777
Epoch 3, Batch 300: Loss = 1.778
Epoch 3, Batch 400: Loss = 1.765
Epoch 3, Batch 500: Loss = 1.773
Epoch 3, Batch 600: Loss = 1.759
Epoch 3, Batch 700: Loss = 1.735
Epoch 4, Batch 100: Loss = 1.717
Epoch 4, Batch 200: Loss = 1.757
Epoch 4, Batch 300: Loss = 1.748
Epoch 4, Batch 400: Loss = 1.724
Epoch 4, Batch 500: Loss = 1.828
Epoch 4, Batch 600: Loss = 1.762
Epoch 4, Batch 700: Loss = 1.746
Epoch 5, Batch 100: Loss = 1.724
Epoch 5, Batch 200: Loss = 1.729
Epoch 5, B

[I 2025-04-25 18:43:32,005] Trial 0 finished with value: 36.47 and parameters: {'num_filters1': 49, 'num_filters2': 71, 'kernel_size': 3, 'dropout_rate': 0.428143814028244, 'fc_units': 320, 'learning_rate': 0.008660716180882034, 'weight_decay': 0.0007630713359372567}. Best is trial 0 with value: 36.47.


Test accuracy: 36.47%
Epoch 1, Batch 100: Loss = 1.875
Epoch 1, Batch 200: Loss = 1.544
Epoch 1, Batch 300: Loss = 1.405
Epoch 1, Batch 400: Loss = 1.317
Epoch 1, Batch 500: Loss = 1.296
Epoch 1, Batch 600: Loss = 1.212
Epoch 1, Batch 700: Loss = 1.171
Epoch 2, Batch 100: Loss = 1.058
Epoch 2, Batch 200: Loss = 1.040
Epoch 2, Batch 300: Loss = 1.023
Epoch 2, Batch 400: Loss = 1.025
Epoch 2, Batch 500: Loss = 0.991
Epoch 2, Batch 600: Loss = 0.975
Epoch 2, Batch 700: Loss = 0.976
Epoch 3, Batch 100: Loss = 0.872
Epoch 3, Batch 200: Loss = 0.850
Epoch 3, Batch 300: Loss = 0.870
Epoch 3, Batch 400: Loss = 0.867
Epoch 3, Batch 500: Loss = 0.831
Epoch 3, Batch 600: Loss = 0.840
Epoch 3, Batch 700: Loss = 0.828
Epoch 4, Batch 100: Loss = 0.736
Epoch 4, Batch 200: Loss = 0.751
Epoch 4, Batch 300: Loss = 0.740
Epoch 4, Batch 400: Loss = 0.754
Epoch 4, Batch 500: Loss = 0.757
Epoch 4, Batch 600: Loss = 0.723
Epoch 4, Batch 700: Loss = 0.708
Epoch 5, Batch 100: Loss = 0.627
Epoch 5, Batch 200: L

[I 2025-04-25 18:47:25,838] Trial 1 finished with value: 72.95 and parameters: {'num_filters1': 83, 'num_filters2': 65, 'kernel_size': 3, 'dropout_rate': 0.2153776032677297, 'fc_units': 287, 'learning_rate': 0.0006419778596852354, 'weight_decay': 5.655021769818772e-05}. Best is trial 1 with value: 72.95.


...

Test accuracy: 65.05%
Epoch 1, Batch 100: Loss = 1.967
Epoch 1, Batch 200: Loss = 1.675
Epoch 1, Batch 300: Loss = 1.555
Epoch 1, Batch 400: Loss = 1.467
Epoch 1, Batch 500: Loss = 1.438
Epoch 1, Batch 600: Loss = 1.382
Epoch 1, Batch 700: Loss = 1.320
Epoch 2, Batch 100: Loss = 1.251
Epoch 2, Batch 200: Loss = 1.240
Epoch 2, Batch 300: Loss = 1.212
Epoch 2, Batch 400: Loss = 1.179
Epoch 2, Batch 500: Loss = 1.168
Epoch 2, Batch 600: Loss = 1.156
Epoch 2, Batch 700: Loss = 1.135
Epoch 3, Batch 100: Loss = 1.082
Epoch 3, Batch 200: Loss = 1.091
Epoch 3, Batch 300: Loss = 1.052
Epoch 3, Batch 400: Loss = 1.044
Epoch 3, Batch 500: Loss = 1.036
Epoch 3, Batch 600: Loss = 1.010
Epoch 3, Batch 700: Loss = 1.010
Epoch 4, Batch 100: Loss = 0.979
Epoch 4, Batch 200: Loss = 0.951
Epoch 4, Batch 300: Loss = 0.949
Epoch 4, Batch 400: Loss = 0.927
Epoch 4, Batch 500: Loss = 0.940
Epoch 4, Batch 600: Loss = 0.928
Epoch 4, Batch 700: Loss = 0.930
Epoch 5, Batch 100: Loss = 0.857
Epoch 5, Batch 200: L

[I 2025-04-25 19:52:48,270] Trial 4 finished with value: 68.62 and parameters: {'num_filters1': 81, 'num_filters2': 195, 'kernel_size': 3, 'dropout_rate': 0.27641765741716373, 'fc_units': 260, 'learning_rate': 0.00014957439385671843, 'weight_decay': 0.00013792020625661206}. Best is trial 1 with value: 72.95.


Test accuracy: 68.62%
Epoch 1, Batch 100: Loss = 2.113
Epoch 1, Batch 200: Loss = 1.847
Epoch 1, Batch 300: Loss = 1.734
Epoch 1, Batch 400: Loss = 1.663
Epoch 1, Batch 500: Loss = 1.619
Epoch 1, Batch 600: Loss = 1.549
Epoch 1, Batch 700: Loss = 1.478
Epoch 2, Batch 100: Loss = 1.427
Epoch 2, Batch 200: Loss = 1.428
Epoch 2, Batch 300: Loss = 1.420
Epoch 2, Batch 400: Loss = 1.379
Epoch 2, Batch 500: Loss = 1.380
Epoch 2, Batch 600: Loss = 1.323
Epoch 2, Batch 700: Loss = 1.334
Epoch 3, Batch 100: Loss = 1.293
Epoch 3, Batch 200: Loss = 1.284
Epoch 3, Batch 300: Loss = 1.277
Epoch 3, Batch 400: Loss = 1.262
Epoch 3, Batch 500: Loss = 1.242
Epoch 3, Batch 600: Loss = 1.242
Epoch 3, Batch 700: Loss = 1.234
Epoch 4, Batch 100: Loss = 1.194
Epoch 4, Batch 200: Loss = 1.206
Epoch 4, Batch 300: Loss = 1.183
Epoch 4, Batch 400: Loss = 1.158
Epoch 4, Batch 500: Loss = 1.158
Epoch 4, Batch 600: Loss = 1.157
Epoch 4, Batch 700: Loss = 1.137
Epoch 5, Batch 100: Loss = 1.120
Epoch 5, Batch 200: L

[I 2025-04-25 19:54:53,915] Trial 5 finished with value: 61.57 and parameters: {'num_filters1': 28, 'num_filters2': 51, 'kernel_size': 3, 'dropout_rate': 0.2500347257251979, 'fc_units': 294, 'learning_rate': 0.0001349934970820153, 'weight_decay': 1.5593956863356024e-05}. Best is trial 1 with value: 72.95.


Test accuracy: 61.57%
Оптимальные гиперпараметры:
num_filters1: 83
num_filters2: 65
kernel_size: 3
dropout_rate: 0.2153776032677297
fc_units: 287
learning_rate: 0.0006419778596852354
weight_decay: 5.655021769818772e-05
Наилучшая точность: 72.95%
